# LeetCode 1328 — Break a Palindrome bằng Python

Link đề: https://leetcode.com/problems/break-a-palindrome/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def breakPalindrome(self, palindrome: str) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu **thứ tự từ điển** (lexicographic order) và vì sao "vị trí sai khác **đầu tiên** quyết định tất cả".
2. Đi từ **brute force** (thử mọi vị trí × mọi chữ cái) → **thu gọn tập ứng viên** → **tham lam `O(n)`**.
3. Nắm hai chi tiết khó: chỉ xét **nửa trái** (ký tự giữa của chuỗi độ dài lẻ không phá được tính đối xứng) và trường hợp **toàn chữ `a`** (đổi ký tự **cuối** thành `b`).
4. Thao tác chuỗi trong Python: chuỗi **bất biến**, đổi một ký tự bằng cắt ghép `s[:i] + c + s[i+1:]` hoặc qua `list`.
5. Tránh lỗi: quên `n == 1`, quét cả chuỗi thay vì nửa trái, đổi ký tự đầu thay vì cuối khi toàn `a`, gán trực tiếp `s[i] = ...`.

## 2. Đề bài & ví dụ

Cho một chuỗi **palindrome** (đọc xuôi ngược như nhau) gồm chữ thường. Hãy thay **đúng một** ký tự bằng một chữ thường bất kỳ sao cho chuỗi mới **không còn là palindrome** và **nhỏ nhất theo thứ tự từ điển**. Trả về chuỗi đó; nếu không có cách nào, trả về chuỗi rỗng `""`.

**Ví dụ 1:** `palindrome = "abccba"` → `"aaccba"` (có nhiều cách như `"zbccba"`, `"aaccba"`, `"abacba"`, nhỏ nhất là `"aaccba"`).

**Ví dụ 2:** `palindrome = "a"` → `""` (một ký tự thì đổi gì vẫn là palindrome).

**Ràng buộc:** `1 <= palindrome.length <= 1000`; chỉ gồm chữ thường tiếng Anh.

*Lưu ý:* "thay" nghĩa là đổi sang một chữ **khác** — thay `'a'` bằng `'a'` thì chuỗi không đổi, vẫn là palindrome nên không bao giờ là đáp án.

In [1]:
import random
import string
import time
from typing import List

examples = [("abccba", "aaccba"), ("a", ""), ("aa", "ab"), ("aba", "abb")]
for s, want in examples:
    print(f"{s!r:10} -> mong đợi {want!r}")

'abccba'   -> mong đợi 'aaccba'
'a'        -> mong đợi ''
'aa'       -> mong đợi 'ab'
'aba'      -> mong đợi 'abb'


## 3. Kiến thức nền

### 3.1 Kiểm tra palindrome

`s == s[::-1]`: `s[::-1]` là chuỗi đảo ngược. Hoặc dùng hai con trỏ `i`, `j = n−1−i` so sánh dần vào giữa.

In [2]:
def is_pal(s):
    return s == s[::-1]

def is_pal_two_pointers(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i, j = i + 1, j - 1
    return True

for s in ["abccba", "aba", "ab", "a", ""]:
    print(repr(s), is_pal(s), is_pal_two_pointers(s))

'abccba' True True
'aba' True True
'ab' False False
'a' True True
'' True True


### 3.2 Thứ tự từ điển

Với hai chuỗi **cùng độ dài**, so sánh ký tự từ trái sang; ở vị trí **khác nhau đầu tiên**, chuỗi có ký tự nhỏ hơn là chuỗi nhỏ hơn. Python so sánh chuỗi đúng theo cách này (theo mã Unicode, với chữ thường thì `'a' < 'b' < … < 'z'`). Hệ quả quan trọng: **làm nhỏ một ký tự càng gần đầu càng có lợi**; ký tự ở sau không cứu được một ký tự lớn hơn ở trước.

In [3]:
print("abcc" < "abcd", "aaz" < "aba", "zaaa" < "azzz")
print(min(["zbccba", "aaccba", "abacba"]))

True True False
aaccba


### 3.3 Chuỗi bất biến — cách đổi một ký tự

Không thể viết `s[i] = 'a'`. Hai cách:
- cắt ghép: `s[:i] + 'a' + s[i+1:]` (tạo chuỗi mới `O(n)`);
- chuyển sang list: `t = list(s); t[i] = 'a'; "".join(t)`.

In [4]:
s = "abccba"
print(s[:1] + "a" + s[2:])
t = list(s)
t[1] = "a"
print("".join(t))
try:
    s[1] = "a"
except TypeError as e:
    print("TypeError:", e)

aaccba
aaccba
TypeError: 'str' object does not support item assignment


### 3.4 Duyệt chữ cái: `string.ascii_lowercase`

`string.ascii_lowercase == "abcdefghijklmnopqrstuvwxyz"` — tiện cho brute force thử mọi chữ.

In [5]:
print(string.ascii_lowercase, len(string.ascii_lowercase))
print([c for c in string.ascii_lowercase if c != "c"][:5])

abcdefghijklmnopqrstuvwxyz 26
['a', 'b', 'd', 'e', 'f']


### 3.5 Vị trí đối xứng và nửa trái

Trong chuỗi độ dài `n`, vị trí `i` đối xứng với `n − 1 − i`. Nửa trái là `i < n // 2`. Với `n` lẻ, vị trí giữa `n // 2` đối xứng **với chính nó** — đổi ký tự giữa thì chuỗi **vẫn** là palindrome.

In [6]:
for s in ["abccba", "abcba"]:
    n = len(s)
    print(s, "| nửa trái:", [(i, s[i]) for i in range(n // 2)], "| giữa:", (n // 2, s[n // 2]) if n % 2 else None)

abccba | nửa trái: [(0, 'a'), (1, 'b'), (2, 'c')] | giữa: None
abcba | nửa trái: [(0, 'a'), (1, 'b')] | giữa: (2, 'c')


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `"abccba"`:

1. Muốn nhỏ nhất ⇒ cố làm ký tự **đầu tiên có thể** nhỏ đi. Ký tự nhỏ nhất là `'a'`.
2. Vị trí 0 đã là `'a'` — đổi thành gì cũng **lớn hơn** ⇒ bỏ qua.
3. Vị trí 1 là `'b'` ⇒ đổi thành `'a'`: `"aaccba"`. Đối xứng của nó (vị trí 4) vẫn là `'b'` ⇒ không còn palindrome ✔. Đây là đáp án.

Tính tay **`"aba"`**: vị trí 0 là `'a'`; vị trí 1 là **ký tự giữa** — đổi `'b'→'a'` cho `"aaa"` vẫn là palindrome ✗. Không làm nhỏ được chỗ nào ⇒ đành làm **lớn** một ký tự, và làm lớn **càng về sau càng ít thiệt**, lớn **ít nhất** có thể: đổi ký tự cuối `'a'→'b'` ⇒ `"abb"` ✔.

Tính tay **`"aa"`**: không có gì để giảm ⇒ đổi ký tự cuối thành `'b'` ⇒ `"ab"` ✔. Còn `"a"`: đổi gì cũng ra chuỗi 1 ký tự (luôn palindrome) ⇒ `""`.

In [7]:
for s in ["abccba", "aba", "aa", "zzz", "aabaa"]:
    n = len(s)
    left = s[: n // 2]
    first = next((i for i, c in enumerate(left) if c != "a"), None)
    print(f"{s!r:9} nửa trái={left!r:6} vị trí khác 'a' đầu tiên ở nửa trái = {first}")

'abccba'  nửa trái='abc'  vị trí khác 'a' đầu tiên ở nửa trái = 1
'aba'     nửa trái='a'    vị trí khác 'a' đầu tiên ở nửa trái = None
'aa'      nửa trái='a'    vị trí khác 'a' đầu tiên ở nửa trái = None
'zzz'     nửa trái='z'    vị trí khác 'a' đầu tiên ở nửa trái = 0
'aabaa'   nửa trái='aa'   vị trí khác 'a' đầu tiên ở nửa trái = None


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử **mọi** vị trí `i` và **mọi** chữ `c ≠ s[i]`, tạo chuỗi mới, nếu không phải palindrome thì so với đáp án tốt nhất. Có `25·n` ứng viên, mỗi ứng viên tốn `O(n)` để tạo và kiểm tra.

In [8]:
def break_palindrome_brute(palindrome):
    best = None
    n = len(palindrome)
    for i in range(n):
        for c in string.ascii_lowercase:
            if c == palindrome[i]:
                continue
            cand = palindrome[:i] + c + palindrome[i + 1:]
            if cand != cand[::-1] and (best is None or cand < best):
                best = cand
    return best if best is not None else ""

for s, want in examples:
    got = break_palindrome_brute(s)
    print(f"{s!r:10} -> {got!r:10} | mong đợi {want!r}", "OK" if got == want else "SAI")

'abccba'   -> 'aaccba'   | mong đợi 'aaccba' OK
'a'        -> ''         | mong đợi '' OK
'aa'       -> 'ab'       | mong đợi 'ab' OK
'aba'      -> 'abb'      | mong đợi 'abb' OK


**Độ phức tạp:** `O(25 · n · n) = O(n²)` thời gian (với `n = 1000` khoảng `2.5·10^7` phép so sánh ký tự — chậm nhưng chạy được), `O(n)` bộ nhớ cho chuỗi tạm.

**Nút thắt:** đa số ứng viên rõ ràng **không thể** là nhỏ nhất: đổi thành chữ lớn hơn `'b'` không bao giờ có lợi, và đổi ở vị trí sau trong khi có thể làm nhỏ ở vị trí trước cũng vô ích. Ta cần **lập luận** để loại bớt — đó là tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ cần xét nửa trái để "làm nhỏ":** đổi một ký tự ở nửa trái thành `'a'` (khi nó khác `'a'`) luôn phá palindrome, vì ký tự đối xứng ở nửa phải vẫn giữ giá trị cũ (≠ `'a'`). Đổi ký tự giữa (n lẻ) không phá được gì. Đổi ở nửa phải thành `'a'` cũng phá được nhưng **luôn kém hơn** đổi ở vị trí đối xứng bên trái (vị trí trái đứng trước).

**Insight 2 — vị trí càng sớm càng tốt:** nếu tồn tại `i` ở nửa trái với `s[i] ≠ 'a'`, chọn `i` **nhỏ nhất** và đặt `s[i] = 'a'`. Mọi ứng viên khác hoặc giữ nguyên vị trí `0..i−1` (đều là `'a'`, không thể nhỏ hơn) và có ký tự `≥ 'a'` ở `i` — hoặc làm **lớn** một ký tự trước `i` ⇒ lớn hơn.

**Insight 3 — khi nửa trái toàn `'a'`:** chuỗi có dạng `aaa…a` hoặc `aa…XaA…a` (chỉ ký tự giữa khác). Không làm nhỏ được ký tự nào có ích ⇒ buộc phải làm **lớn** một ký tự. Làm lớn ở vị trí **cuối cùng** (ít ảnh hưởng nhất), lên **`'b'`** (lớn ít nhất). Ký tự cuối ở nửa phải và là `'a'` (đối xứng với vị trí 0) ⇒ đổi thành `'b'` chắc chắn phá palindrome (khi `n ≥ 2`).

**Quy tắc "nếu… thì…":**
- Nếu `n == 1` ⇒ `""`.
- Nếu có `i < n // 2` với `s[i] != 'a'` ⇒ đổi `s[i]` (nhỏ nhất) thành `'a'`.
- Ngược lại ⇒ đổi `s[n−1]` thành `'b'`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thu gọn tập ứng viên (bước trung gian)

Từ insight, chỉ cần hai loại ứng viên: (a) đổi một ký tự `≠ 'a'` thành `'a'`; (b) đổi một ký tự `'a'` thành `'b'`. Ta vẫn thử **mọi vị trí** nhưng chỉ với hai chữ đó ⇒ `2n` ứng viên thay vì `25n`. Cách này giúp ta **kiểm chứng trực giác** trước khi tin hẳn vào tham lam.

In [9]:
def break_palindrome_candidates(palindrome):
    n = len(palindrome)
    cands = []
    for i in range(n):
        c = "a" if palindrome[i] != "a" else "b"
        cand = palindrome[:i] + c + palindrome[i + 1:]
        if cand != cand[::-1]:
            cands.append(cand)
    return min(cands) if cands else ""

for s, want in examples:
    got = break_palindrome_candidates(s)
    print(f"{s!r:10} -> {got!r:10}", "OK" if got == want else "SAI")

'abccba'   -> 'aaccba'   OK
'a'        -> ''         OK
'aa'       -> 'ab'       OK
'aba'      -> 'abb'      OK


Vì sao chỉ cần `'a'` hoặc `'b'`? Tại một vị trí cố định `i`, nếu `s[i] ≠ 'a'` thì `'a'` cho chuỗi nhỏ nhất (và luôn phá palindrome nếu `i` không phải giữa). Nếu `s[i] == 'a'` thì chữ khác nhỏ nhất là `'b'`. Code nộp LeetCode (Cách 2):

In [10]:
class SolutionCandidates:
    def breakPalindrome(self, palindrome: str) -> str:
        best = ""
        for i, ch in enumerate(palindrome):
            cand = palindrome[:i] + ("a" if ch != "a" else "b") + palindrome[i + 1:]
            if cand != cand[::-1] and (best == "" or cand < best):
                best = cand
        return best

print([SolutionCandidates().breakPalindrome(s) for s, _ in examples])

['aaccba', '', 'ab', 'abb']


**Độ phức tạp:** `O(n²)` thời gian (`n` ứng viên × `O(n)` mỗi ứng viên), `O(n)` bộ nhớ. Nhanh hơn brute 25 lần nhưng chưa tuyến tính.

### 7.2 Cách 3 — Tham lam `O(n)` (lời giải chuẩn)

**Từng đoạn:**
1. `n = len(s)`; nếu `n == 1` trả `""`.
2. Duyệt `i` từ `0` đến `n // 2 − 1`: gặp `s[i] != 'a'` ⇒ trả `s[:i] + 'a' + s[i+1:]` ngay.
3. Ra khỏi vòng lặp (nửa trái toàn `'a'`) ⇒ trả `s[:-1] + 'b'`.

In [11]:
def break_palindrome_greedy(palindrome):
    n = len(palindrome)
    if n == 1:
        return ""
    for i in range(n // 2):
        if palindrome[i] != "a":
            return palindrome[:i] + "a" + palindrome[i + 1:]
    return palindrome[:-1] + "b"

for s, want in examples:
    got = break_palindrome_greedy(s)
    print(f"{s!r:10} -> {got!r:10}", "OK" if got == want else "SAI")

'abccba'   -> 'aaccba'   OK
'a'        -> ''         OK
'aa'       -> 'ab'       OK
'aba'      -> 'abb'      OK


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionGreedy:
    def breakPalindrome(self, palindrome: str) -> str:
        n = len(palindrome)
        if n == 1:
            return ""
        for i in range(n // 2):
            if palindrome[i] != "a":
                return palindrome[:i] + "a" + palindrome[i + 1:]
        return palindrome[:-1] + "b"

print([SolutionGreedy().breakPalindrome(s) for s, _ in examples])

['aaccba', '', 'ab', 'abb']


**Độ phức tạp:** vòng lặp tối đa `n/2` bước, cắt ghép một lần `O(n)` ⇒ `O(n)` thời gian, `O(n)` bộ nhớ cho chuỗi kết quả (bắt buộc vì phải trả về chuỗi mới).

### 7.3 Cách 4 — Tham lam viết bằng list (phong cách "sửa tại chỗ")

Cùng ý tưởng, nhưng chuyển sang `list` để "sửa tại chỗ" như trong C++/Java, rồi `join`. Hữu ích khi muốn đổi nhiều ký tự hoặc dịch sang ngôn ngữ khác.

In [13]:
def break_palindrome_list(palindrome):
    if len(palindrome) == 1:
        return ""
    chars = list(palindrome)
    for i in range(len(chars) // 2):
        if chars[i] != "a":
            chars[i] = "a"
            return "".join(chars)
    chars[-1] = "b"
    return "".join(chars)

for s, want in examples:
    print(f"{s!r:10} -> {break_palindrome_list(s)!r}")

'abccba'   -> 'aaccba'
'a'        -> ''
'aa'       -> 'ab'
'aba'      -> 'abb'


## 8. TRACE từng bước

Trace tham lam trên ba chuỗi:

| chuỗi | n | nửa trái (i < n//2) | i dừng | hành động | kết quả |
|---|---|---|---|---|---|
| `"abccba"` | 6 | `a b c` | 1 (`'b'`) | `s[1] = 'a'` | `"aaccba"` |
| `"aba"` | 3 | `a` | — | đổi cuối → `'b'` | `"abb"` |
| `"aaaa"` | 4 | `a a` | — | đổi cuối → `'b'` | `"aaab"` |
| `"a"` | 1 | — | — | `n == 1` | `""` |

In [14]:
def trace_break(palindrome):
    n = len(palindrome)
    print(f"s={palindrome!r}, n={n}, n//2={n // 2}")
    if n == 1:
        print("  n == 1 -> ''")
        return ""
    for i in range(n // 2):
        ch = palindrome[i]
        print(f"  i={i}: s[i]={ch!r}, đối xứng s[{n-1-i}]={palindrome[n-1-i]!r} -> {'đổi thành a' if ch != 'a' else 'đã là a, đi tiếp'}")
        if ch != "a":
            res = palindrome[:i] + "a" + palindrome[i + 1:]
            print("  kết quả:", repr(res))
            return res
    res = palindrome[:-1] + "b"
    print(f"  nửa trái toàn 'a' -> đổi s[{n-1}] thành 'b': {res!r}")
    return res

for s in ["abccba", "aba", "aaaa", "a"]:
    trace_break(s)

s='abccba', n=6, n//2=3
  i=0: s[i]='a', đối xứng s[5]='a' -> đã là a, đi tiếp
  i=1: s[i]='b', đối xứng s[4]='b' -> đổi thành a
  kết quả: 'aaccba'
s='aba', n=3, n//2=1
  i=0: s[i]='a', đối xứng s[2]='a' -> đã là a, đi tiếp
  nửa trái toàn 'a' -> đổi s[2] thành 'b': 'abb'
s='aaaa', n=4, n//2=2
  i=0: s[i]='a', đối xứng s[3]='a' -> đã là a, đi tiếp
  i=1: s[i]='a', đối xứng s[2]='a' -> đã là a, đi tiếp
  nửa trái toàn 'a' -> đổi s[3] thành 'b': 'aaab'
s='a', n=1, n//2=0
  n == 1 -> ''


Trace brute force trên `"aba"` — liệt kê ứng viên **hợp lệ nhỏ nhất** ở từng vị trí, để thấy vì sao đáp án là `"abb"`:

In [15]:
def trace_brute_per_position(palindrome):
    n = len(palindrome)
    for i in range(n):
        valid = []
        for c in string.ascii_lowercase:
            if c == palindrome[i]:
                continue
            cand = palindrome[:i] + c + palindrome[i + 1:]
            if cand != cand[::-1]:
                valid.append(cand)
        print(f"vị trí {i}: {len(valid)} ứng viên hợp lệ, nhỏ nhất = {min(valid) if valid else None!r}")

trace_brute_per_position("aba")

vị trí 0: 25 ứng viên hợp lệ, nhỏ nhất = 'bba'
vị trí 1: 0 ứng viên hợp lệ, nhỏ nhất = None
vị trí 2: 25 ứng viên hợp lệ, nhỏ nhất = 'abb'


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def breakPalindrome(self, palindrome: str) -> str:
        n = len(palindrome)
        if n == 1:                                   # một ký tự: không thể phá
            return ""
        for i in range(n // 2):                      # chỉ nửa trái (bỏ ký tự giữa)
            if palindrome[i] != "a":
                return palindrome[:i] + "a" + palindrome[i + 1:]
        return palindrome[:-1] + "b"                 # nửa trái toàn 'a'

for s, want in examples:
    got = Solution().breakPalindrome(s)
    print(f"{s!r:10} -> {got!r:10}", "OK" if got == want else "SAI")

'abccba'   -> 'aaccba'   OK
'a'        -> ''         OK
'aa'       -> 'ab'       OK
'aba'      -> 'abb'      OK


## 10. Vì sao thuật toán đúng

**Trường hợp `n = 1`:** mọi chuỗi một ký tự là palindrome ⇒ không có đáp án ⇒ `""`.

**Trường hợp có `i < n/2` với `s[i] ≠ 'a'` (lấy `i` nhỏ nhất):** chuỗi `t` thu được hợp lệ vì `t[i] = 'a' ≠ s[n−1−i] = t[n−1−i]` (vị trí đối xứng khác `i` do `i < n/2`). Xét một đáp án hợp lệ bất kỳ `u` (đổi vị trí `j` thành `c`):
- nếu `j < i`: `s[j] = 'a'` nên `c > 'a'` ⇒ `u` lớn hơn `s` tại `j`, còn `t` bằng `s` tại `j` ⇒ `t < u`;
- nếu `j = i`: `u[i] = c ≥ 'a' = t[i]`, phần còn lại giống nhau ⇒ `t ≤ u`;
- nếu `j > i`: `u[i] = s[i] > 'a' = t[i]`, các vị trí trước bằng nhau ⇒ `t < u`.

Vậy `t` nhỏ nhất.

**Trường hợp nửa trái toàn `'a'` (`n ≥ 2`):** khi đó nửa phải cũng toàn `'a'` (đối xứng); chỉ ký tự giữa (nếu có) có thể khác. Đổi một ký tự thành chữ nhỏ hơn là không thể (nếu là `'a'`) hoặc vô ích (ký tự giữa ⇒ vẫn palindrome). Nên mọi đáp án hợp lệ đều đổi một ký tự `'a'` ở vị trí `j ≠ giữa` thành `c ≥ 'b'`. Chuỗi `t = s[:-1] + 'b'` hợp lệ (vì `t[0] = 'a' ≠ 'b' = t[n−1]`). Với đáp án `u` khác: nếu `j < n−1` thì `u` lớn hơn `s` tại `j` trong khi `t` bằng `s` tại `j` ⇒ `t < u`; nếu `j = n−1` thì `c ≥ 'b'` ⇒ `t ≤ u`. Vậy `t` nhỏ nhất.

## 11. Phân tích Time / Space

Đặt `n = len(palindrome)` (`≤ 1000`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(25·n²)` | `O(n)` | 25n ứng viên × kiểm tra `O(n)` |
| 2. Thu gọn ứng viên | `O(n²)` | `O(n)` | n ứng viên |
| 3. Tham lam | `O(n)` | `O(n)` (chuỗi kết quả) | chuẩn |
| 4. Tham lam (list) | `O(n)` | `O(n)` | giống Cách 3 |

Không thể dưới `O(n)` vì phải tạo chuỗi kết quả độ dài `n`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
solvers = {
    "brute": break_palindrome_brute,
    "candidates": break_palindrome_candidates,
    "greedy": break_palindrome_greedy,
    "list": break_palindrome_list,
    "submit": lambda s: Solution().breakPalindrome(s),
    "submit_cand": lambda s: SolutionCandidates().breakPalindrome(s),
    "submit_greedy": lambda s: SolutionGreedy().breakPalindrome(s),
}
cases = examples + [
    ("z", ""),               # một ký tự khác 'a'
    ("bb", "ab"),            # n chẵn nhỏ
    ("aaa", "aab"),          # toàn a, lẻ
    ("aaaa", "aaab"),        # toàn a, chẵn
    ("aabaa", "aabab"),      # chỉ giữa khác a -> đổi cuối
    ("zaz", "aaz"),          # vị trí đầu khác a
    ("abba", "aaba"),
    ("racecar", "aacecar"),
]
for s, want in cases:
    for name, f in solvers.items():
        assert f(s) == want, (name, s, f(s), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 12 case qua cho 7 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh palindrome ngẫu nhiên bằng cách tạo nửa trái + (ký tự giữa nếu lẻ) + đảo ngược nửa trái. Dùng bảng chữ nhỏ `{a, b, c}` để thường xuyên có nhiều chữ `a` (trường hợp khó).

In [18]:
def random_palindrome(rng, n, alphabet="abc"):
    half = "".join(rng.choice(alphabet) for _ in range(n // 2))
    mid = rng.choice(alphabet) if n % 2 else ""
    return half + mid + half[::-1]

rng = random.Random(1328)
for _ in range(2000):
    n = rng.randint(1, 12)
    s = random_palindrome(rng, n, rng.choice(["a", "ab", "abc", string.ascii_lowercase]))
    assert s == s[::-1]
    res = {name: f(s) for name, f in solvers.items()}
    assert len(set(res.values())) == 1, (s, res)
    out = res["submit"]
    if out:                                           # kiểm tra hợp lệ độc lập
        assert len(out) == len(s) and sum(a != b for a, b in zip(out, s)) == 1 and out != out[::-1]
print("2000 palindrome ngẫu nhiên: mọi cách khớp brute force, kết quả hợp lệ")

2000 palindrome ngẫu nhiên: mọi cách khớp brute force, kết quả hợp lệ


### 12.3 Kích thước lớn (`n = 999, 1000`)

In [19]:
big_cases = ["a" * 1000, "a" * 499 + "b" + "a" * 499, "a" * 499 + "bb" + "a" * 499, random_palindrome(random.Random(9), 1000)]
for s in big_cases:
    assert s == s[::-1]
    t0 = time.time(); g = Solution().breakPalindrome(s); tg = time.time() - t0
    t0 = time.time(); b = break_palindrome_brute(s); tb = time.time() - t0
    diff = [i for i in range(len(s)) if s[i] != g[i]]
    print(f"giống brute: {g == b} | vị trí đổi {diff} | tham lam {tg*1000:.2f}ms, brute {tb*1000:.0f}ms")

giống brute: True | vị trí đổi [999] | tham lam 0.01ms, brute 24ms


giống brute: True | vị trí đổi [998] | tham lam 0.01ms, brute 23ms
giống brute: True | vị trí đổi [499] | tham lam 0.01ms, brute 24ms
giống brute: True | vị trí đổi [0] | tham lam 0.00ms, brute 22ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quét cả chuỗi thay vì nửa trái (quên ký tự giữa)

In [20]:
def wrong_full_scan(palindrome):
    if len(palindrome) == 1:
        return ""
    for i in range(len(palindrome)):                 # SAI: bao gồm cả ký tự giữa
        if palindrome[i] != "a":
            return palindrome[:i] + "a" + palindrome[i + 1:]
    return palindrome[:-1] + "b"

for s in ["aba", "aabaa"]:
    print(f"{s!r}: sai={wrong_full_scan(s)!r} (palindrome? {is_pal(wrong_full_scan(s))}) | đúng={Solution().breakPalindrome(s)!r}")

'aba': sai='aaa' (palindrome? True) | đúng='abb'
'aabaa': sai='aaaaa' (palindrome? True) | đúng='aabab'


**Vì sao sai:** với `"aba"`, ký tự khác `'a'` đầu tiên là **ký tự giữa**; đổi nó thành `'a'` ra `"aaa"` — **vẫn palindrome**. (Quét cả chuỗi còn có thể chạm vào nửa phải, nhưng trước đó luôn gặp vị trí đối xứng ở nửa trái nên chỉ ký tự giữa gây lỗi.) **Sửa:** `for i in range(n // 2)`.

### Lỗi 2 — Toàn chữ `a` thì đổi ký tự **đầu** thành `'b'`

In [21]:
def wrong_first_b(palindrome):
    n = len(palindrome)
    if n == 1:
        return ""
    for i in range(n // 2):
        if palindrome[i] != "a":
            return palindrome[:i] + "a" + palindrome[i + 1:]
    return "b" + palindrome[1:]                      # SAI

for s in ["aa", "aaaa", "aba"]:
    print(f"{s!r}: sai={wrong_first_b(s)!r} | đúng={Solution().breakPalindrome(s)!r}")

'aa': sai='ba' | đúng='ab'
'aaaa': sai='baaa' | đúng='aaab'
'aba': sai='bba' | đúng='abb'


**Vì sao sai:** `"baaa"` hợp lệ nhưng **không nhỏ nhất** — ký tự đầu tăng lên làm cả chuỗi lớn. Làm lớn ở vị trí **cuối** cho `"aaab" < "baaa"`. **Sửa:** `palindrome[:-1] + "b"`.

### Lỗi 3 — Quên trường hợp `n == 1`

In [22]:
def wrong_no_len_check(palindrome):
    n = len(palindrome)
    for i in range(n // 2):
        if palindrome[i] != "a":
            return palindrome[:i] + "a" + palindrome[i + 1:]
    return palindrome[:-1] + "b"

for s in ["a", "z"]:
    print(f"{s!r}: sai={wrong_no_len_check(s)!r} (vẫn là palindrome!) | đúng={Solution().breakPalindrome(s)!r}")

'a': sai='b' (vẫn là palindrome!) | đúng=''
'z': sai='b' (vẫn là palindrome!) | đúng=''


**Vì sao sai:** với `n = 1`, vòng lặp không chạy, code trả `"b"` — một chuỗi một ký tự, **vẫn là palindrome** ⇒ không hợp lệ, đề yêu cầu `""`. **Sửa:** kiểm tra `if n == 1: return ""` ở đầu.

### Lỗi 4 — Gán trực tiếp vào chuỗi

In [23]:
def wrong_assign(palindrome):
    if len(palindrome) == 1:
        return ""
    for i in range(len(palindrome) // 2):
        if palindrome[i] != "a":
            palindrome[i] = "a"                      # SAI: str không hỗ trợ gán
            return palindrome
    return palindrome[:-1] + "b"

try:
    print(wrong_assign("abccba"))
except TypeError as e:
    print("lỗi:", type(e).__name__, "-", e)
print("đúng:", Solution().breakPalindrome("abccba"))

lỗi: TypeError - 'str' object does not support item assignment
đúng: aaccba


**Vì sao sai:** chuỗi Python **bất biến**. **Sửa:** cắt ghép `s[:i] + "a" + s[i+1:]` hoặc dùng `list` rồi `join` (Cách 4).

### Lỗi 5 — Đổi ký tự cuối thành `'b'` mà không kiểm tra nửa trái trước

In [24]:
def wrong_always_last(palindrome):
    if len(palindrome) == 1:
        return ""
    return palindrome[:-1] + ("b" if palindrome[-1] == "a" else "a")

s = "abccba"
print(f"sai={wrong_always_last(s)!r} | đúng={Solution().breakPalindrome(s)!r}")

sai='abccbb' | đúng='aaccba'


**Vì sao sai:** `"abccbb"` hợp lệ nhưng lớn hơn `"aaccba"` (khác nhau ở vị trí 1: `'b' > 'a'`). Ưu tiên **làm nhỏ sớm** trước, chỉ khi không thể mới làm lớn ở cuối.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Nhỏ nhất theo thứ tự từ điển**" + "thay đổi ít" ⇒ tham lam **từ trái sang**: làm nhỏ vị trí sớm nhất có thể; nếu buộc phải làm lớn thì làm ở **vị trí muộn nhất**, tăng **ít nhất**.
- Palindrome ⇒ chỉ cần xét **nửa trái**; nhớ ký tự **giữa** khi độ dài lẻ.
- Luôn kiểm tra các trường hợp biên: `n = 1`, toàn cùng một chữ, chỉ ký tự giữa khác.
- Nghi ngờ tham lam? Viết brute force thử mọi vị trí × mọi chữ, đối chiếu với palindrome ngẫu nhiên trên bảng chữ nhỏ.

## 15. Biến thể của bài

1. **Lớn nhất theo thứ tự từ điển** (Bài tập 4): đổi ký tự khác `'z'` đầu tiên ở nửa trái thành `'z'`; nếu toàn `'z'` thì đổi ký tự cuối thành `'y'`.
2. **Đếm số chuỗi không palindrome** thu được bằng một lần thay (Bài tập 5).
3. **LeetCode 1842 — Next Palindrome Using Same Digits** (Premium): palindrome lớn hơn kế tiếp.
4. **LeetCode 680 — Valid Palindrome II:** xoá tối đa một ký tự để thành palindrome (chiều ngược lại).
5. **LeetCode 1704/2697 — Lexicographically Smallest Palindrome:** sửa ít ký tự nhất để **thành** palindrome nhỏ nhất: với mỗi cặp đối xứng, lấy `min` của hai ký tự.

In [25]:
def make_smallest_palindrome(s):
    """LeetCode 2697: thay ít nhất để thành palindrome, ưu tiên nhỏ nhất."""
    t = list(s)
    i, j = 0, len(t) - 1
    while i < j:
        t[i] = t[j] = min(t[i], t[j])
        i, j = i + 1, j - 1
    return "".join(t)

print(make_smallest_palindrome("egcfe"), make_smallest_palindrome("abcd"), make_smallest_palindrome("seven"))

efcfe abba neven


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi vị trí × 25 chữ, giữ nhỏ nhất hợp lệ | `O(25·n²)` | `O(n)` | kiểm chứng, khi chưa nghĩ ra tham lam |
| 2. Thu gọn ứng viên | mỗi vị trí chỉ thử `'a'` hoặc `'b'` | `O(n²)` | `O(n)` | bước trung gian, dễ chứng minh |
| 3. Tham lam | khác `'a'` đầu tiên ở nửa trái → `'a'`; không có → cuối thành `'b'` | `O(n)` | `O(n)` | lời giải nộp |
| 4. Tham lam (list) | như 3, sửa trên list | `O(n)` | `O(n)` | dịch sang ngôn ngữ có chuỗi khả biến |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().breakPalindrome("aabaa")` và `Solution().breakPalindrome("bab")` trả về gì?

### Bài 2 (tính tay)
Tính tay cho `"zz"`, `"azza"`, `"aaazaaa"`. Ghi rõ nhánh nào của thuật toán được dùng.

### Bài 3 (điền TODO)
Hoàn thiện `break_todo`.

### Bài 4 (tự viết lại — biến thể)
Viết `break_palindrome_max(s)`: thay đúng một ký tự để chuỗi **không** palindrome và **lớn nhất** theo thứ tự từ điển (hoặc `""`). Đối chiếu với brute force.

### Bài 5 (mini challenge)
Viết `count_breaks(s)` = số chuỗi **khác nhau** không phải palindrome thu được bằng cách thay đúng một ký tự (bằng một chữ khác). Tìm công thức `O(1)` theo `n` và kiểm chứng bằng brute force.

In [26]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def break_todo(palindrome):
    n = len(palindrome)
    # TODO: trường hợp n == 1
    # TODO: duyệt nửa trái, gặp ký tự khác 'a' thì đổi
    # TODO: còn lại đổi ký tự cuối thành 'b'
    pass

def break_palindrome_max_todo(s):
    # TODO
    pass

def count_breaks_todo(s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `"aabaa"`: nửa trái `"aa"`. (2) Toàn `'a'` ⇒ nhánh đổi cuối. (3) `"bab"`: nửa trái `"b"` ⇒ đổi vị trí 0.

**Bài 2:** (1) `"zz"`: nửa trái `"z"`. (2) `"azza"`: nửa trái `"az"`, dừng ở `i = 1`. (3) `"aaazaaa"`: nửa trái `"aaa"` toàn `'a'`, `'z'` là ký tự giữa ⇒ đổi cuối.

**Bài 3:** (1) `if n == 1: return ""`. (2) `for i in range(n // 2)`. (3) `return palindrome[:-1] + "b"`.

**Bài 4:** (1) Đối xứng vai trò `'a'` ↔ `'z'`. (2) Nửa trái tìm ký tự khác `'z'` đầu tiên ⇒ `'z'`. (3) Không có ⇒ ký tự cuối (`'z'`) thành `'y'`.

**Bài 5:** (1) Có `25·n` cách thay (mỗi vị trí 25 chữ khác), các chuỗi thu được đều khác nhau. (2) Thay ở vị trí `i` mà vẫn palindrome ⇔ `i` là vị trí giữa. (3) Đáp án `25·n − (25 nếu n lẻ, ngược lại 0)`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
print("Bài 1:", Solution().breakPalindrome("aabaa"), Solution().breakPalindrome("bab"))
assert Solution().breakPalindrome("aabaa") == "aabab"
assert Solution().breakPalindrome("bab") == "aab"
b2 = [Solution().breakPalindrome(s) for s in ["zz", "azza", "aaazaaa"]]
print("Bài 2:", b2)
assert b2 == ["az", "aaza", "aaazaab"]

# Bài 3
def break_sol(palindrome):
    n = len(palindrome)
    if n == 1:
        return ""
    for i in range(n // 2):
        if palindrome[i] != "a":
            return palindrome[:i] + "a" + palindrome[i + 1:]
    return palindrome[:-1] + "b"

for s, want in cases:
    assert break_sol(s) == want
print("Bài 1–3: qua")

Bài 1: aabab aab
Bài 2: ['az', 'aaza', 'aaazaab']
Bài 1–3: qua


In [28]:
# Bài 4
def break_palindrome_max(s):
    n = len(s)
    if n == 1:
        return ""
    for i in range(n // 2):
        if s[i] != "z":
            return s[:i] + "z" + s[i + 1:]
    return s[:-1] + "y"

def break_palindrome_max_brute(s):
    best = ""
    for i in range(len(s)):
        for c in string.ascii_lowercase:
            if c != s[i]:
                cand = s[:i] + c + s[i + 1:]
                if cand != cand[::-1] and cand > best:
                    best = cand
    return best

# Bài 5
def count_breaks(s):
    n = len(s)
    return 25 * n - (25 if n % 2 else 0)

def count_breaks_brute(s):
    seen = set()
    for i in range(len(s)):
        for c in string.ascii_lowercase:
            if c != s[i]:
                cand = s[:i] + c + s[i + 1:]
                if cand != cand[::-1]:
                    seen.add(cand)
    return len(seen)

rng = random.Random(4)
for _ in range(600):
    s = random_palindrome(rng, rng.randint(1, 10), rng.choice(["z", "yz", "xyz", "abz"]))
    assert break_palindrome_max(s) == break_palindrome_max_brute(s), s
    assert count_breaks(s) == count_breaks_brute(s), s
print("Bài 4:", break_palindrome_max("zzz"), break_palindrome_max("abba"), "| Bài 5:", count_breaks("aba"), count_breaks("abba"))
print("Bài 4–5: 600 palindrome ngẫu nhiên khớp brute force")

Bài 4: zzy zbba | Bài 5: 50 100
Bài 4–5: 600 palindrome ngẫu nhiên khớp brute force


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam "phá palindrome nhỏ nhất":**

```python
n = len(s)
if n == 1:
    return ""
for i in range(n // 2):          # nửa trái, bỏ ký tự giữa
    if s[i] != 'a':
        return s[:i] + 'a' + s[i+1:]
return s[:-1] + 'b'              # toàn 'a' (trừ có thể ký tự giữa)
```

**Nguyên tắc từ điển:** làm **nhỏ** ở vị trí **sớm nhất**; nếu buộc làm **lớn** thì ở vị trí **muộn nhất**, tăng **ít nhất**.

**Checklist trước khi submit:**
- [ ] `n == 1` ⇒ `""`.
- [ ] Vòng lặp `range(n // 2)` (không đụng ký tự giữa).
- [ ] Nhánh toàn `'a'`: đổi ký tự **cuối** thành `'b'`.
- [ ] Không gán trực tiếp vào chuỗi.
- [ ] Đã thử `"aba"`, `"aa"`, `"a"`, `"aabaa"`.
- [ ] Đúng chữ ký: `breakPalindrome(self, palindrome: str) -> str`.